In [29]:
%load_ext autoreload
%autoreload 2

import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import random

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model

sys.path.append(os.path.dirname(os.getcwd()))
from notepad import TraceDataGeneration
#from TraceDataGeneration_single_trace import TraceDataGeneration

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
function_list = ['constant',
                 'exponential_form',
                 'exponential_pulse',
                 'impulse_like',
                 'linear_transition',
                 'rectangular_pulse',
                 'step_like',
                 'trapezoidal',
                 'triangular']

In [20]:
function = []
for i in range(2):
    function.append(random.choice(function_list))

In [21]:
function

['linear_transition', 'constant']

In [2]:
functions=['linear_transition', 'step_like']

In [41]:
generator = TraceDataGeneration(n=10, global_range=0)
data = generator.automated_generation_random_para(step_num=4)

In [47]:
# Trace 생성
generator = TraceDataGeneration(n=5, global_range=100)
num_traces = 10  # 생성할 trace 수

traces = []
for _ in range(num_traces):
    trace = generator.automated_generation_random_para(step_num=5, jitter=True)
    traces.append(trace)

In [162]:
traces = []
params = []
for i in range(3):
    generator = TraceDataGeneration(n=5, global_range=1, seed=i)
    random_trace= generator.automated_generation_random_para(function=functions)
    traces.append(random_trace)
    print('\n')

{'start_value': -11.20446599600729, 'end_value': 10.0, 't1': 30, 't2': 59, 'length': 69}
{'start_value': 95.0, 'end_value': -11.20446599600729, 't1': 31, 'length': 39}


{'start_value': 5.0, 'end_value': 10.0, 't1': 16, 't2': 25, 'length': 39}
{'start_value': 95.0, 'end_value': 5.0, 't1': 11, 'length': 20}


{'start_value': 81.79864595797505, 'end_value': 100.0, 't1': 6, 't2': 8, 'length': 20}
{'start_value': 2.8011725311889855, 'end_value': 81.79864595797505, 't1': 2, 'length': 43}




In [177]:
a={'start_value': -11.20446599600729, 'end_value': 10.0, 't1': 30, 't2': 59, 'length': 69}
b={'start_value': 95.0, 'end_value': -11.20446599600729, 't1': 31, 'length': 39}


c={'start_value': 5.0, 'end_value': 10.0, 't1': 16, 't2': 25, 'length': 39}
d={'start_value': 95.0, 'end_value': 5.0, 't1': 11, 'length': 20}


e={'start_value': 81.79864595797505, 'end_value': 100.0, 't1': 6, 't2': 8, 'length': 20}
f={'start_value': 2.8011725311889855, 'end_value': 81.79864595797505, 't1': 2, 'length': 43}

In [178]:
y_pred = []
for idx, values in a.items():
    y_pred.append(values)
for idx, values in b.items():
    y_pred.append(values)
for idx, values in c.items():
    y_pred.append(values)
for idx, values in d.items():
    y_pred.append(values)
for idx, values in e.items():
    y_pred.append(values)
for idx, values in f.items():
    y_pred.append(values)

In [179]:
y_pred = np.array(y_pred).reshape(-1,9)

In [32]:
def make_windowform(data, window_size, offset=1):
    windows = []
    ts = data

    for i in range(0, data.shape[0] - window_size + 1, offset): # +1을 추가해줘야 마지막 끝 데이터까지 감
        windows.append(np.array(ts[i: i + window_size]))

    windows = np.array(windows)
    #print(f"number of samples : {len(windows)} / samples' shape: {windows.shape}")

    return windows

In [50]:
def data_to_tensor(trainx, batch_size, mode='train'):
    #dataset = trainx.reshape(-1,1)
    dataset = trainx
    print("dataset shape : ", dataset.shape)
    if mode == "test":
        return dataset
    # Create TensorFlow dataset
    train_ds = tf.data.Dataset.from_tensor_slices(dataset)
    train_ds = (train_ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE))
    return train_ds

In [61]:
window_size = 10
batch_size = 16
offset = 3

In [62]:
window = make_windowform(data=random_trace[0].PARAMETER_VALUE.values, window_size=window_size, offset=offset)

In [45]:
window.shape

(25, 10)

In [63]:
#tensor_x = data_to_tensor(window, batch_size=batch_size)

dataset shape :  (25, 10)


In [86]:
def generator_model():
    inputs = keras.layers.Input(shape=window_size)
    x = keras.layers.Dense(20, activation='relu')(inputs)
    x = keras.layers.Dense(20, activation='relu')(x)
    outputs = keras.layers.Dense(7)(x)
    
    model = Model(inputs, outputs)
    return model

In [87]:
model = generator_model()
model.summary()

Model: "model_5"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_7 (InputLayer)         [(None, 10)]              0         
_________________________________________________________________
dense_18 (Dense)             (None, 20)                220       
_________________________________________________________________
dense_19 (Dense)             (None, 20)                420       
_________________________________________________________________
dense_20 (Dense)             (None, 8)                 168       
Total params: 808
Trainable params: 808
Non-trainable params: 0
_________________________________________________________________


In [191]:
tstart_value, tend_value, tt1, tt2, tlength, sstart_value, send_value, st1, slength = y_pred[:10,:]

ValueError: not enough values to unpack (expected 9, got 3)

In [189]:
tstart_value

-11.20446599600729

In [ ]:
def big_loss(y_pred):
    def mse_loss(y_true, y_pred):
        tstart_value, tend_value, tt1, tt2, tlength, sstart_value, send_value, st1, slength = y_pred
        
        tt1 = int(tt1)
        tt2 = int(tt2)
        tlength = int(tlength)
        
        st1 = int(st1)
        slength = int(slength)
        
        

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(), loss='mse')